# Домашняя работа 4. Классификация изнутри

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 4 — Линейная классификация: логистическая регрессия, метрики и SVM |
| Опора | материал семинара 4 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии всё считал `sklearn`. Дома напишем сами: логистическую регрессию с градиентом и гессианом из лекции 3 (и сравним градиентный спуск с методом Ньютона), ROC-кривую с проверкой её вероятностной интерпретации и, наконец, двойственную задачу SVM — ту самую, из которой берутся опорные векторы.

Первые две задачи выполняются на **вашей** выборке; для двойственной задачи нужна маленькая и наглядная выборка, поэтому там берётся синтетика — зато решение проверяется точно.

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from labdata import load_personal
from scipy import optimize
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=4)
describe_variant(variant)

In [ ]:
# Ваша выборка. Занятие про классификацию, поэтому регрессионный вариант
# бинаризуем по медиане обучающей части («дороже медианы» / «дешевле»).
data = load_personal(variant)
Xd, yd = np.vstack([data["X_train"], data["X_test"]]), \
         np.r_[data["y_train"], data["y_test"]]
if data["task"] == "regression":
    yd = (yd > np.median(data["y_train"])).astype(int)
    print(f"вариант регрессионный -> бинаризован по медиане: "
          f"«{data['target']} выше медианы» = класс 1")
yd = yd.astype(int)

print(f"{data['domain']}: {Xd.shape[0]} объектов, {Xd.shape[1]} признаков, "
      f"доля класса 1 = {yd.mean():.3f}")

---
# Задача 1. Логистическая регрессия своими руками

Утверждение 3.4: для $Q(\theta) = \sum_i\mathcal L(a_\theta,x_i,y_i)$

$$
\nabla_\theta Q = X^{\mathsf T}(h - y), \qquad h = \sigma(X\theta),
$$

а по утверждению 3.6 гессиан равен $\nabla^2 Q = X^{\mathsf T}SX$, где
$S = \mathrm{diag}\bigl(h_i(1-h_i)\bigr)$. Наличие гессиана позволяет применить
**метод Ньютона** (в статистике — IRLS):

$$
\theta^{(t+1)} = \theta^{(t)} - \bigl(X^{\mathsf T}SX\bigr)^{-1}X^{\mathsf T}(h-y).
$$

In [ ]:
def sigmoid(z):
    """Численно устойчивая сигмоида (как на занятии)."""
    raise NotImplementedError


def logloss(theta, X, y, l2=0.0):
    """Q(theta) -- сумма логистических потерь + (l2/2)||theta_{1:}||^2.

    Устойчиво считается как np.logaddexp(0, z) - y * z, где z = X @ theta.
    """
    raise NotImplementedError


def logloss_grad(theta, X, y, l2=0.0):
    """Градиент X^T (h - y) по утв. 3.4; свободный член не регуляризуем."""
    raise NotImplementedError


def logloss_hess(theta, X, y, l2=0.0):
    """Гессиан X^T S X по утв. 3.6, S = diag(h_i (1 - h_i))."""
    raise NotImplementedError

### Задание 1.1. Обязательная проверка формул

Приёмом из домашней работы 2 проверьте и градиент, и гессиан конечными
разностями. Гессиан — это якобиан градиента, так что проверяется так же.

In [ ]:
Xc = np.column_stack([np.ones(80), rng.normal(size=(80, 4))])
yc = (rng.random(80) < sigmoid(Xc @ np.array([0.3, 1.0, -1.0, 0.5, 0.0]))).astype(float)
th0 = rng.normal(size=5) * 0.3

# TODO: проверьте градиент и гессиан центральными разностями (h = 1e-6).
#       Гессиан -- якобиан градиента, считается так же, только от logloss_grad.

### Задание 1.2. Градиентный спуск против метода Ньютона

In [ ]:
def fit_gd(X, y, n_iter=3000, l2=1.0):
    """Градиентный спуск с шагом eta = 4/||X||_2^2 (константа Липшица)."""
    raise NotImplementedError


def fit_newton(X, y, n_iter=12, l2=1.0):
    """Метод Ньютона: theta -= H^{-1} g (используйте np.linalg.solve, не inv)."""
    raise NotImplementedError


# TODO: разбейте со стратификацией, отмасштабируйте, добавьте столбец единиц,
#       обучите обоими методами (l2=1.0) и сравните с
#       LogisticRegression(C=1.0): значение Q, коэффициенты, число итераций.

In [ ]:
# TODO: постройте график сходимости обоих методов (Q - Q* в лог. масштабе,
#       по оси итераций достаточно первых 60).

> **Вывод.** Сколько итераций потребовалось каждому методу? Почему разница такая большая и почему тогда метод Ньютона не используют всегда?
>
> *(ваш ответ здесь)*

---
# Задача 2. ROC-кривая своими руками

ROC строится по всем порогам сразу, и считать её перебором порогов не нужно:
достаточно отсортировать объекты по убыванию балла. Тогда $TP(t)$ и $FP(t)$ —
это кумулятивные суммы меток.

Полезная интерпретация, которую мы проверим: **AUC равна вероятности того, что
случайный объект класса 1 получит больший балл, чем случайный объект класса 0.**

In [ ]:
def roc_curve_manual(y_true, score):
    """Точки ROC-кривой без перебора порогов.

    Подсказка: отсортируйте объекты по УБЫВАНИЮ балла; тогда np.cumsum(y)
    даёт TP, а np.cumsum(1 - y) -- FP. Нормируйте на общее число
    положительных и отрицательных, добавьте начальную точку (0, 0).
    """
    raise NotImplementedError


def auc_manual(fpr, tpr):
    """Площадь под кривой методом трапеций."""
    raise NotImplementedError


# TODO: посчитайте баллы на контрольной выборке своей моделью из задачи 1
#       и сверьте свой AUC со sklearn.metrics.roc_auc_score.

In [ ]:
# TODO: проверьте интерпретацию AUC методом Монте-Карло: возьмите 200000 пар
#       (случайный объект класса 1, случайный объект класса 0) и посчитайте
#       долю пар, где балл положительного больше. Сравните с AUC.
# TODO: постройте свою ROC-кривую.

> **Вывод.** Совпала ли доля пар с AUC? Почему ROC не зависит от доли классов, а PR-кривая — зависит?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 3★. Двойственная задача SVM

Теорема 3.11: двойственная задача —

$$
\max_{\alpha}\ \sum_i\alpha_i - \frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j (x_i^{\mathsf T}x_j)
\quad\text{при}\quad \alpha_i\ge0,\ \ \sum_i\alpha_i y_i = 0,
$$

а затем $w^*=\sum_i\alpha_i^*y_ix_i$, и $b^*$ восстанавливается по любому объекту
с $\alpha_i^*>0$: $b^* = y_i - w^{*\mathsf T}x_i$.

Решать будем как задачу минимизации $\frac12\alpha^{\mathsf T}P\alpha - \mathbf 1^{\mathsf T}\alpha$,
где $P = (yy^{\mathsf T})\odot K$, с ограничениями $0\le\alpha_i\le C$ и $\alpha^{\mathsf T}y=0$.

In [ ]:
def solve_dual(K, y, C=None):
    """Двойственная задача через scipy.optimize.minimize (SLSQP).

    Минимизируется 0.5 a^T P a - sum(a), где P = (y y^T) * K,
    при 0 <= a_i <= C (bounds) и a^T y = 0 (constraints, type='eq').
    Передайте jac -- сходимость будет заметно быстрее.
    """
    raise NotImplementedError


def recover_wb(alpha, X, y, tol=1e-6):
    """w = sum a_i y_i x_i; b -- среднее (y_i - w^T x_i) по опорным векторам."""
    raise NotImplementedError

In [ ]:
from sklearn.svm import SVC

Xb, yb = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=RANDOM_STATE)
yb = np.where(yb == 0, -1.0, 1.0)
Xb = StandardScaler().fit_transform(Xb) * 1.2

# TODO: решите двойственную задачу с линейным ядром K = Xb @ Xb.T,
#       восстановите (w, b), сравните с SVC(kernel="linear", C=1e6),
#       посчитайте число опорных векторов и выведите наибольшие alpha.

### Задание 3.1. Ядровой трюк

И двойственная задача, и итоговый классификатор зависят от объектов **только
через скалярные произведения**. Значит, заменив $x_i^{\mathsf T}x_j$ на $K(x_i,x_j)$,
мы получаем нелинейную границу, не меняя алгоритм:

$$
a(x) = \mathrm{sign}\Bigl(\sum_i\alpha_i^*y_i K(x_i,x) + b^*\Bigr).
$$

Соберите такой классификатор с RBF-ядром и проверьте его на `make_circles`.

In [ ]:
def rbf(A, B, gamma=1.0):
    """K(x, x') = exp(-gamma ||x - x'||^2); расстояния -- как в занятии 1."""
    raise NotImplementedError


from sklearn.datasets import make_circles
Xc, yc = make_circles(n_samples=150, noise=0.12, factor=0.45, random_state=RANDOM_STATE)
yc = np.where(yc == 0, -1.0, 1.0)
Xc = StandardScaler().fit_transform(Xc)

# TODO: решите двойственную задачу с матрицей K = rbf(Xc, Xc), C = 1;
#       восстановите b по опорным векторам, соберите decision(Z) по формуле выше,
#       посчитайте точность и нарисуйте границу решения с отмеченными
#       опорными векторами.

> **Вывод.** Сколько объектов оказалось опорными и почему у RBF-ядра их больше, чем у линейного? Почему в формуле классификатора нигде не появляется $\varphi(x)$?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Ядро $K(x,x')=\|x-x'\|$ не удовлетворяет критерию Мерсера. Что конкретно сломается, если подставить его в двойственную задачу?
2. Вы обучили SVM, и опорными оказались 95 % объектов. О чём это говорит и какой гиперпараметр вы будете менять в первую очередь?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.